In [ ]:
import os, subprocess, sys, traceback

print("== 1. pip install kaggle-benchmarks ==")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle-benchmarks"],
                   capture_output=True, text=True)
print("rc", r.returncode)

print("== 2. arreglar el conflicto de protobuf (gencode 5.29.6 > runtime 5.29.5) ==")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "protobuf>=5.29.6"],
                   capture_output=True, text=True)
print("rc", r.returncode, (r.stderr or "")[-200:])

print("== 3. import ==")
kbench = None
try:
    import kaggle_benchmarks as kbench
    print("import OK", getattr(kbench, "__version__", "sin version"))
except Exception as exc:
    print("import fallo:", type(exc).__name__, str(exc)[:300])

if kbench is not None:
    print("\n== 4. que modelos hay en kbench.llms? ==")
    try:
        llms = kbench.llms
        keys = list(llms.keys()) if hasattr(llms, "keys") else [str(k) for k in llms]
        print("modelos:", keys[:25], "... total", len(keys))
    except Exception:
        traceback.print_exc()

    print("\n== 5. credenciales del proxy presentes? (solo NOMBRES, nunca valores) ==")
    names = sorted(k for k in os.environ
                   if any(t in k.upper() for t in ("KAGGLE", "PROXY", "MODEL", "GEMINI", "API")))
    print("variables:", names[:20])

    print("\n== 6. llamada real ==")
    try:
        answer = kbench.llm.prompt("Reply with exactly one word: WORKS")
        print("respuesta:", repr(answer)[:300])
    except Exception as exc:
        print("fallo la llamada:", type(exc).__name__, str(exc)[:400])
